# 03 - Signals and trades, up close

What a strategy sees and does in a window of the TEST block: price with every entry, exit, take-profit and stop;
the three horizons' P(up) against the entry lines; confidence and strength; predicted sigma with the variance
spikes; equity and drawdown, all as the window's own numbers. Set `START` to a bar number, or to
"steepest_fall" / "worst_trade" / "last", and `BARS` to the window length.

In [1]:
# Parameters
RUN_DIR = None
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"
STRATEGY = "calibrated_quantile"
START = "worst_trade"           # a bar number, or "steepest_fall" / "worst_trade" / "last"
BARS = 600

In [2]:
import os
from pathlib import Path

from IPython.display import display

from neural_trade.notebook import BacktestExplorer, pick_run
from neural_trade.visualization.trading_dashboard import detail_window

run_dir = pick_run(RUN_DIR, RUNS_DIR)   # newest run with a serving bundle, or a clear error
print("run:", run_dir)
explorer = BacktestExplorer.from_run(run_dir, csv_path=CSV_PATH)
res = explorer.run(STRATEGY, costs={"random_seeds": 0}, baselines=False)
start = detail_window(res, BARS, around=START)[0] if isinstance(START, str) else int(START)
explorer.dashboard(res, start=start, end=start + BARS).show()

run: ..\runs\20260924T165923Z-fa75177-dirty-af67ee43
CalibrationPipeline loaded from '..\runs\20260924T165923Z-fa75177-dirty-af67ee43\artifacts\calibration/'


Dataset length after cleaning: 43500


Date range after cleaning: 2025-10-11 02:30:00+00:00 to 2025-11-10 07:29:00+00:00


## The signal features over the whole block

Shares of bars for the boolean flags, consensus and horizon votes, and the distribution of the numeric features.

In [3]:
for name, table in explorer.signal_summary().items():
    print(name)
    display(table.round(4))

features


,count,mean,std,min,25%,50%,75%,max
weighted_direction,7236.0,0.4947,0.0242,0.3660,0.4826,0.4953,0.5080,0.6817
weighted_move_$,7236.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
strength,7236.0,0.0560,0.0374,0.0007,0.0311,0.0466,0.0704,0.3782
avg_confidence,7236.0,0.5049,0.1720,0.0528,0.3626,0.5254,0.6592,0.7807
agreement,7236.0,0.3618,0.1026,0.3333,0.3333,0.3333,0.3333,1.0000
volatility_$,7236.0,218.1407,56.7515,130.2792,168.7312,209.6423,260.7563,429.5721


flags (share of bars true)


,share true,bars
magnitude_coherent,1.0000,7236
direction_aligned,0.1696,7236
var_spike,0.0159,7236


consensus


,share of bars
up,0.0992
down,0.1924
neutral,0.7084


horizon votes (P > 0.55 or < 0.45)


,share of bars
0 votes,0.6748
1 vote,0.2143
2+ votes,0.1108


confidence scale


,var_scale
confidence = exp(-var / var_scale); var_scale = median predicted variance on the CAL block,1.0694


## Trades

In [4]:
explorer.trade_analytics(res).show()
trades = res.trades_frame()
if len(trades):
    display(trades.groupby("exit_reason")[["net_pnl", "gross_pnl", "bars_held"]].agg(["count", "mean", "sum"]).round(2))
    display(trades.groupby("side")[["net_pnl", "gross_pnl"]].agg(["count", "sum", "mean"]).round(2))
trades.tail(30).round(2)

net_pnl                 gross_pnl                bars_held         \
              count   mean      sum     count   mean     sum     count   mean   
exit_reason                                                                     
REV             132 -16.83 -2221.72       132   4.69  618.98       132   6.36   
SL                5 -68.19  -340.94         5 -47.46 -237.32         5   5.40   
TIME             30 -25.86  -775.72        30  -5.56 -166.65        30  15.00   

                  
             sum  
exit_reason       
REV          839  
SL            27  
TIME         450

net_pnl                 gross_pnl             
        count      sum   mean     count    sum  mean
side                                                
LONG       76 -1528.44 -20.11        76  113.7  1.50
SHORT      91 -1809.93 -19.89        91  101.3  1.11

,side,entry_bar,exit_bar,entry_price,exit_price,notional,gross_pnl,costs,exit_reason,tp,sl,entry_reason,net_pnl,bars_held,return_pct
137,LONG,4769,4784,101815.98,101551.68,7148.31,-14.28,18.57,TIME,None,101418.40,quantile,-32.84,15,-0.46
138,LONG,4853,4860,101610.47,101600.55,7115.46,3.58,18.50,REV,None,101149.21,quantile,-14.93,7,-0.21
139,LONG,5087,5097,101817.81,101761.61,7100.54,0.34,18.46,REV,None,101336.45,quantile,-18.12,10,-0.26
140,LONG,5127,5138,102012.58,102040.19,7082.41,6.17,18.42,REV,None,101464.07,quantile,-12.25,11,-0.17
141,SHORT,5386,5401,101646.79,101957.83,7070.16,-17.38,18.41,TIME,None,102459.99,quantile,-35.79,15,-0.51
142,SHORT,5405,5417,101910.25,101968.58,7034.37,0.20,18.29,REV,None,102572.81,quantile,-18.09,12,-0.26
143,SHORT,5975,5990,101577.48,101693.75,7016.28,-3.82,18.25,TIME,None,102163.66,quantile,-22.07,15,-0.31
144,SHORT,6235,6250,103598.68,103337.98,6994.21,21.79,18.16,TIME,None,104257.39,quantile,3.63,15,0.05
145,LONG,6256,6259,103078.82,103247.47,6997.84,15.66,18.21,REV,None,102459.98,quantile,-2.56,3,-0.04
146,SHORT,6262,6265,103182.45,103015.09,6995.28,15.54,18.17,REV,None,103868.13,quantile,-2.63,3,-0.04
